In [1]:
from folde.data import get_proteingym_dataset
dms_id = 'FLIP-AAV'

wt_aa_seq, naturalness_df, embedding_df, activity_df, category_df = get_proteingym_dataset(
    dms_id,
    '300m',
    '600m',
)

/Users/jacobroberts/git/foldy/backend/src/folde/data.py:179: DtypeWarning: Columns (12,14,16,22) have mixed types. Specify dtype option on import or set low_memory=False.
  incomplete_activity_df = pd.read_csv(FLIP_AAV_DATA_FILE)
/Users/jacobroberts/git/foldy/backend/src/folde/data.py:222: DtypeWarning: Columns (12,14,16,22) have mixed types. Specify dtype option on import or set low_memory=False.
  category_df = pd.read_csv(FLIP_AAV_DATA_FILE)
/Users/jacobroberts/git/foldy/backend/src/folde/data.py:230: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  category_df = category_df.fillna(False)


In [2]:
from folde.few_shot_models import get_few_shot_model

configs = {
    "RandomForestFewShotModel": {
        "n_estimators": 100,
        "criterion": "friedman_mse",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "min_weight_fraction_leaf": 0.0,
        "max_features": 1.0,
        "max_leaf_nodes": None,
        "min_impurity_decrease": 0.0,
        "bootstrap": True,
        "oob_score": False,
        "n_jobs": None,
        "verbose": 0,
        "warm_start": False,
        "ccp_alpha": 0.0,
        "max_samples": None,
        "random_state": 42,
        "wt_aa_seq": wt_aa_seq,
    },
    "TorchMLPFewShotModel": {
        "pretrain": True,
        "pretrain_epochs": 50,
        "ensemble_size": 5,
        "embedding_dim": 960,
        "hidden_dims": [100, 50],
        "dropout": 0.2,
        "learning_rate": 3e-4,
        "weight_decay": 1e-5,
        "train_epochs": 200,
        "train_patience": 40,
        "val_frequency": 10,
        "do_validation_with_pair_fraction": 0.2,
        "decision_mode": "constantliar",
        "lie_noise_stddev_multiplier": 2.0,
        "random_state": 42,
        "wt_aa_seq": wt_aa_seq,
    },
}

models = {}
for model_name, model_params in configs.items():
    models[model_name] = get_few_shot_model(model_name, **model_params)

In [3]:
from app.helpers.sequence_util import is_homolog_seq_id, get_loci_set

naturalness_series = naturalness_df.wt_marginal
embedding_series = embedding_df.embedding
activity_series = activity_df.DMS_score

def is_single_mutant_id(seq_id: str) -> bool:
    if seq_id == 'WT' or is_homolog_seq_id(seq_id):
        return False
    return len(get_loci_set(seq_id)) == 1
single_mutant_seq_ids = [seq_id for seq_id in naturalness_series.index if is_single_mutant_id(seq_id)]
pretraining_naturalness_series = naturalness_series.loc[single_mutant_seq_ids]
pretraining_embedding_series = embedding_series.loc[single_mutant_seq_ids]

for model_name, model in models.items():
    model.pretrain(
        pretraining_naturalness_series,
        pretraining_embedding_series,
    )


/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/torch/cuda/amp/grad_scaler.py:126: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warn

In [ ]:
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from folde.util import get_consensus_scores, get_top_percentile_recall_score, top_k_mask
from tqdm.contrib.concurrent import thread_map

results = {}



for i, (model_name, model) in enumerate(tqdm(models.items(), desc="Models", position=0)):
    for j, benchmark in enumerate(tqdm(
        ['one_vs_many_split', 'two_vs_many_split', 'seven_vs_many_split', 'low_vs_high_split', 'mut_des_split'],
        desc="Benchmarks", position=1, leave=False
    )):
        is_train_row = category_df[benchmark]
        train_seq_ids = category_df[is_train_row].index
        held_out_seq_ids = category_df[~is_train_row].index

        model.fit(
            naturalness_series.loc[train_seq_ids],
            embedding_series.loc[train_seq_ids],
            activity_series.loc[train_seq_ids],
            None, None, None
        )

        predictions = model.predict(
            naturalness_series.loc[held_out_seq_ids],
            embedding_series.loc[held_out_seq_ids],
        )

        mean_prediction = sum(predictions) / len(predictions)

        held_out_activity_series = activity_series.loc[held_out_seq_ids]
        activity_is_nonnull = held_out_activity_series.notna()
        
        results[(model_name, benchmark, 'spearman')] = spearmanr(
            mean_prediction[activity_is_nonnull],
            held_out_activity_series[activity_is_nonnull]
        )
        
        results[(model_name, benchmark, 'recall1pct')] = get_top_percentile_recall_score(
            mean_prediction[activity_is_nonnull].to_numpy(),
            held_out_activity_series[activity_is_nonnull].to_numpy(),
            1,
        )
        
        results[(model_name, benchmark, 'recall10pct')] = get_top_percentile_recall_score(
            mean_prediction[activity_is_nonnull].to_numpy(),
            held_out_activity_series[activity_is_nonnull].to_numpy(),
            10,
        )

results

Models:   0%|          | 0/2 [00:00<?, ?it/s]

Benchmarks:   0%|          | 0/5 [00:00<?, ?it/s]